# Day 04: Mixture of Experts (MoE) Routing
> *Inference Engineering* — Chapter 2.2.4 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime
**Prerequisite:** Day 02 (transformer blocks, FFN)

**Topics:** Dense vs sparse models, expert routing, active vs total parameters, inference implications

## Overview

In Day 02 we saw that the **feed-forward network (FFN)** is the largest component of each
transformer block. Mixture of Experts (MoE) is an architecture that makes the FFN **sparse**:
instead of one big FFN, the model has many smaller ones (the "experts"), and a **router**
picks which experts to activate for each token.

The infra analogy: it's like **microservices vs a monolith**. A dense model is a monolith —
every request goes through the same code. An MoE model is a microservices architecture —
a router decides which services handle each request, and most services stay idle for
any given request.

The key insight from the book: a model like Qwen3-235B has 235 billion total parameters,
but only **22 billion are active per token**. This means:
- The model needs GPU **memory** for all 235B params (they all must be loaded)
- But it only needs **compute** for 22B params per forward pass
- Single-request latency is great (less compute), but memory requirements are high

This notebook builds MoE routing from scratch so you can see how it works.

In [ ]:
# Setup
!pip install -q numpy torch matplotlib

import numpy as np
import matplotlib.pyplot as plt
import math
import time

try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
    print(f"PyTorch {torch.__version__}")
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not available -- numpy-only mode")

plt.rcParams.update({
    "figure.facecolor": "#0d1117", "axes.facecolor": "#161b22",
    "axes.edgecolor": "#30363d",   "text.color": "#e6edf3",
    "axes.labelcolor": "#e6edf3",  "xtick.color": "#8b949e",
    "ytick.color": "#8b949e",      "grid.color": "#21262d",
})
%matplotlib inline

## Part 1: Dense vs Sparse — The Core Idea

A **dense** FFN is a single big matrix multiply. Every input token goes through
the same weights. A **sparse** MoE FFN has N smaller matrices (experts), and only
K of them are activated per token (typically K=2 out of N=64 or N=128).

Let's build both and compare.

In [ ]:
# Dense FFN vs MoE FFN -- built from scratch

np.random.seed(42)
d_model = 256       # hidden dimension
d_ff    = 1024      # FFN inner dimension for dense model
n_experts = 8       # number of experts in MoE
top_k     = 2       # experts activated per token

# Each expert is a smaller FFN: d_model -> d_expert -> d_model
d_expert = d_ff // n_experts  # 128 (each expert is 1/8th the size of the dense FFN)

print("=== Dense FFN ===")
dense_up   = np.random.randn(d_model, d_ff) * 0.02     # 256 -> 1024
dense_down = np.random.randn(d_ff, d_model) * 0.02     # 1024 -> 256
dense_params = dense_up.size + dense_down.size
print(f"Weights: up={dense_up.shape}, down={dense_down.shape}")
print(f"Total params: {dense_params:,}")
print(f"Active params per token: {dense_params:,} (100% -- all weights used)")

print(f"\n=== MoE FFN ({n_experts} experts, top-{top_k} routing) ===")
experts_up   = [np.random.randn(d_model, d_expert) * 0.02 for _ in range(n_experts)]
experts_down = [np.random.randn(d_expert, d_model) * 0.02 for _ in range(n_experts)]
router_weights = np.random.randn(d_model, n_experts) * 0.02  # tiny router network

moe_params = sum(e.size for e in experts_up) + sum(e.size for e in experts_down) + router_weights.size
active_params = (experts_up[0].size + experts_down[0].size) * top_k + router_weights.size
print(f"Per-expert weights: up={experts_up[0].shape}, down={experts_down[0].shape}")
print(f"Router weights: {router_weights.shape}")
print(f"Total params: {moe_params:,}")
print(f"Active params per token: {active_params:,} ({active_params/moe_params*100:.0f}%)")
print(f"\nMoE has {moe_params/dense_params:.1f}x the total params but only uses {active_params/dense_params*100:.0f}% of the dense cost per token.")

## Part 2: The Router — How Tokens Pick Their Experts

The **router** is a tiny network (just a linear layer + softmax) that runs on each token
and outputs a probability distribution over the experts. The top-K experts are selected
and their outputs are weighted by the router probabilities.

From the book: in Qwen's architecture with 128 experts, the router picks 8 experts
at each of the 94 layers for every token. The routing is **per-token, per-layer** —
different tokens in the same batch can go to different experts.

In [ ]:
# Implement MoE routing from scratch

def moe_route(token_hidden, router_weights, top_k=2):
    """Route a single token to its top-k experts."""
    # Step 1: compute router logits (one score per expert)
    logits = token_hidden @ router_weights  # [d_model] @ [d_model, n_experts] -> [n_experts]

    # Step 2: softmax to get probabilities
    exp_logits = np.exp(logits - logits.max())
    probs = exp_logits / exp_logits.sum()

    # Step 3: pick top-k experts
    top_indices = np.argsort(probs)[-top_k:][::-1]
    top_probs   = probs[top_indices]
    # Renormalize so the selected expert weights sum to 1
    top_probs   = top_probs / top_probs.sum()

    return top_indices, top_probs, probs

def moe_forward(token_hidden, router_weights, experts_up, experts_down, top_k=2):
    """Full MoE forward pass for one token."""
    indices, weights, all_probs = moe_route(token_hidden, router_weights, top_k)

    # Run only the selected experts and combine their outputs
    output = np.zeros(d_model)
    for idx, weight in zip(indices, weights):
        # Each expert: up-project -> ReLU -> down-project
        hidden = token_hidden @ experts_up[idx]         # d_model -> d_expert
        hidden = np.maximum(hidden, 0)                   # ReLU
        expert_out = hidden @ experts_down[idx]          # d_expert -> d_model
        output += weight * expert_out                     # weighted sum

    return output, indices, weights, all_probs

# Run routing for several different tokens
print("=== Routing 6 different tokens through 8 experts (top-2) ===")
print(f"{'Token':<10} {'Expert 1':>10} {'Weight 1':>10} {'Expert 2':>10} {'Weight 2':>10}")
print("-" * 55)

token_labels = ["The", "server", "crashed", "due", "to", "memory"]
all_routing = []
for i, label in enumerate(token_labels):
    np.random.seed(i * 7 + 3)
    token_vec = np.random.randn(d_model) * 0.5
    _, indices, weights, probs = moe_forward(token_vec, router_weights, experts_up, experts_down, top_k=2)
    all_routing.append(probs)
    print(f"{label:<10} {f'Expert {indices[0]}':>10} {weights[0]:>10.3f} {f'Expert {indices[1]}':>10} {weights[1]:>10.3f}")

print("\nDifferent tokens route to different experts.")
print("This is why MoE inference needs all experts in memory even though only a few are active per token.")

In [ ]:
# Visualize routing: heatmap of expert selection probabilities

routing_matrix = np.array(all_routing)  # [n_tokens, n_experts]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Left: full routing probabilities
im = ax1.imshow(routing_matrix, cmap='viridis', aspect='auto')
ax1.set_yticks(range(len(token_labels)))
ax1.set_yticklabels(token_labels)
ax1.set_xticks(range(n_experts))
ax1.set_xticklabels([f'E{i}' for i in range(n_experts)])
ax1.set_xlabel('Expert', color='#e6edf3')
ax1.set_ylabel('Token', color='#e6edf3')
ax1.set_title('Router probabilities per token\n(brighter = higher probability)', color='#e6edf3')
plt.colorbar(im, ax=ax1)

# Right: expert utilization (how often each expert is in top-2)
expert_counts = np.zeros(n_experts)
np.random.seed(0)
for i in range(200):  # simulate 200 tokens
    token_vec = np.random.randn(d_model) * 0.5
    _, indices, _, _ = moe_forward(token_vec, router_weights, experts_up, experts_down, top_k=2)
    for idx in indices:
        expert_counts[idx] += 1

colors = ['#da3633' if c > 60 or c < 40 else '#238636' for c in expert_counts]
ax2.bar(range(n_experts), expert_counts, color=colors, alpha=0.85)
ax2.axhline(200 * top_k / n_experts, color='#f0883e', ls='--', alpha=0.8, label=f'Ideal: {200*top_k/n_experts:.0f} each')
ax2.set_xlabel('Expert', color='#e6edf3')
ax2.set_ylabel('Times selected (out of 200 tokens)', color='#e6edf3')
ax2.set_title('Expert utilization (load balance)\nRed = imbalanced', color='#e6edf3')
ax2.set_xticks(range(n_experts))
ax2.set_xticklabels([f'E{i}' for i in range(n_experts)])
ax2.legend(framealpha=0.2)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("Load balancing is a real challenge in MoE -- if one expert gets overloaded,")
print("it becomes a bottleneck (same as a hot shard in a distributed database).")
print("MoE training uses auxiliary losses to encourage balanced routing.")

## Part 3: Memory vs Compute Tradeoff

This is the key insight for infrastructure planning:

| | Dense (Llama-3-70B) | MoE (Qwen3-235B) |
|---|---|---|
| Total params | 70B | 235B |
| Active params/token | 70B | ~22B |
| GPU memory needed | ~140 GB | ~470 GB |
| Compute per token | 70B ops | 22B ops |
| Single-request latency | Higher | Lower |

MoE gives you a **bigger brain** (235B params) at the **compute cost** of a smaller model (22B active).
But you pay for it in **memory** — all experts must be loaded.

There's also a catch for batched serving that the book highlights: in production,
different requests activate different experts, so with enough concurrent requests,
**all experts are hot**. The sparsity benefit applies per-token, not per-batch.

In [ ]:
# Simulate: what fraction of experts are active as batch size grows?

n_experts_sim = 128  # like Qwen
top_k_sim     = 8    # like Qwen
batch_sizes   = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]

fractions = []
np.random.seed(42)
for bs in batch_sizes:
    active = set()
    for _ in range(bs):
        # Each token in the batch picks top_k random experts
        # In reality routing depends on the token, but the effect is similar
        selected = np.random.choice(n_experts_sim, size=top_k_sim, replace=False)
        active.update(selected)
    fractions.append(len(active) / n_experts_sim * 100)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(batch_sizes, fractions, 'o-', color='#da3633', lw=2, markersize=8)
ax.axhline(100, color='#f0883e', ls='--', alpha=0.7, label='100% = all experts active (dense equivalent)')
ax.set_xlabel('Batch size (concurrent requests)', color='#e6edf3')
ax.set_ylabel('% of experts active in batch', color='#e6edf3')
ax.set_title('MoE sparsity vanishes with batch size\n(128 experts, top-8 routing)', color='#e6edf3')
ax.set_xscale('log', base=2)
ax.legend(framealpha=0.2)
ax.grid(alpha=0.3)
ax.set_ylim(0, 110)

# Annotate key points
for bs, frac in zip(batch_sizes, fractions):
    if bs in [1, 8, 64, 512]:
        ax.annotate(f'{frac:.0f}%', (bs, frac), textcoords='offset points',
                    xytext=(0, 12), ha='center', color='#e6edf3', fontsize=9)

plt.tight_layout()
plt.show()
print(f"Batch=1:   {fractions[0]:.0f}% of experts active (great sparsity -- fast single request)")
print(f"Batch=64:  {fractions[batch_sizes.index(64)]:.0f}% active (most experts needed)")
print(f"Batch=512: {fractions[-1]:.0f}% active (nearly all -- memory AND compute are full)")
print(f"\nThis is why MoE models excel at low-batch/interactive inference")
print(f"but their advantage shrinks under heavy batch load.")

## Part 4: Dense vs MoE — Compute Comparison

Let's measure the actual compute difference between a dense forward pass
and an MoE forward pass on the same sized model.

In [ ]:
# Time dense vs MoE forward pass

d_model    = 512
d_ff_dense = 2048           # dense FFN inner dim
n_experts  = 8
d_expert   = d_ff_dense // n_experts  # 256 per expert
top_k      = 2
n_tokens   = 64             # simulate a batch of tokens

np.random.seed(0)
tokens = np.random.randn(n_tokens, d_model).astype(np.float32)

# Dense FFN weights
W_up   = np.random.randn(d_model, d_ff_dense).astype(np.float32) * 0.02
W_down = np.random.randn(d_ff_dense, d_model).astype(np.float32) * 0.02

# MoE expert weights
E_up   = [np.random.randn(d_model, d_expert).astype(np.float32) * 0.02 for _ in range(n_experts)]
E_down = [np.random.randn(d_expert, d_model).astype(np.float32) * 0.02 for _ in range(n_experts)]
R      = np.random.randn(d_model, n_experts).astype(np.float32) * 0.02

# Time dense
t0 = time.perf_counter()
for _ in range(100):
    dense_out = np.maximum(tokens @ W_up, 0) @ W_down
dense_ms = (time.perf_counter() - t0) / 100 * 1000

# Time MoE
t0 = time.perf_counter()
for _ in range(100):
    moe_out = np.zeros_like(tokens)
    # Route each token
    logits = tokens @ R
    exp_l = np.exp(logits - logits.max(axis=-1, keepdims=True))
    probs = exp_l / exp_l.sum(axis=-1, keepdims=True)
    top_experts = np.argsort(probs, axis=-1)[:, -top_k:]  # [n_tokens, top_k]

    for expert_idx in range(n_experts):
        # Find which tokens selected this expert
        mask = np.any(top_experts == expert_idx, axis=-1)  # [n_tokens]
        if not mask.any():
            continue
        selected = tokens[mask]
        expert_out = np.maximum(selected @ E_up[expert_idx], 0) @ E_down[expert_idx]
        # Simplified: equal weighting for timing purposes
        moe_out[mask] += expert_out / top_k
moe_ms = (time.perf_counter() - t0) / 100 * 1000

dense_flops = n_tokens * (d_model * d_ff_dense + d_ff_dense * d_model)
moe_flops   = n_tokens * (d_model * d_expert + d_expert * d_model) * top_k

print(f"{'':20s} {'Dense':>12} {'MoE':>12}")
print("-" * 48)
print(f"{'Total params':20s} {W_up.size + W_down.size:>12,} {sum(e.size for e in E_up) + sum(e.size for e in E_down) + R.size:>12,}")
print(f"{'Active params/tok':20s} {W_up.size + W_down.size:>12,} {(E_up[0].size + E_down[0].size) * top_k:>12,}")
print(f"{'FLOPs ({n_tokens} tokens)':20s} {dense_flops:>12,} {moe_flops:>12,}")
print(f"{'Time (ms)':20s} {dense_ms:>12.2f} {moe_ms:>12.2f}")
print(f"\nMoE uses {moe_flops/dense_flops*100:.0f}% of the dense FLOPs but has routing overhead.")
print(f"At scale (billions of params), the FLOP savings dominate the routing cost.")

## Part 5: When to Use MoE

The book notes these practical rules:

- **Models > 100B params** strongly benefit from MoE (Qwen3-235B, Mixtral, DeepSeek-V3)
- **Models < 32B** and especially < 8B tend to use dense architectures efficiently
- **Domain-specific models** (code completion, tab completion) don't gain much — the model acts as effectively one expert
- **MoE unlocks Expert Parallelism** — different experts on different GPUs, covered in Day 15

For infrastructure planning, the key question is:
**Do I have enough GPU memory to hold all experts, even though most are idle per request?**

In [ ]:
# Memory planning: dense vs MoE for real models

models = [
    # (name, total_params_B, active_params_B, is_moe)
    ("Llama-3.1-8B",      8,    8,    False),
    ("Llama-3.1-70B",     70,   70,   False),
    ("Mixtral-8x7B",      47,   13,   True),
    ("Mixtral-8x22B",     141,  39,   True),
    ("DeepSeek-V3",       671,  37,   True),
    ("Qwen3-235B",        235,  22,   True),
]

print(f"{'Model':<20} {'Total':>8} {'Active':>8} {'Memory':>10} {'Compute':>10} {'Type':>6}")
print(f"{'':20s} {'(B)':>8} {'(B)':>8} {'FP16 GB':>10} {'rel.':>10}")
print("-" * 65)
for name, total, active, is_moe in models:
    mem_gb = total * 2  # FP16: 2 bytes per param
    mtype = "MoE" if is_moe else "Dense"
    print(f"{name:<20} {total:>8} {active:>8} {mem_gb:>9.0f}GB {active:>9}B  {mtype:>6}")

print(f"\nDeepSeek-V3: 671B params = ~1.3 TB of GPU memory just for weights.")
print(f"But per-token compute is only 37B -- comparable to a dense 40B model.")
print(f"\nThis is why large MoE models need multi-GPU (memory) but serve fast (compute).")

# GPU memory comparison
fig, ax = plt.subplots(figsize=(10, 5))
names = [m[0] for m in models]
mem    = [m[1] * 2 for m in models]  # FP16 GB
active = [m[2] * 2 for m in models]
x = np.arange(len(names))
w = 0.35

ax.bar(x - w/2, mem,    w, label='Memory needed (all params)', color='#da3633', alpha=0.85)
ax.bar(x + w/2, active, w, label='Compute cost (active params)', color='#238636', alpha=0.85)
ax.axhline(80,  color='#f0883e', ls='--', alpha=0.7, label='80 GB (1x A100)')
ax.axhline(640, color='#1f6feb', ls='--', alpha=0.7, label='640 GB (8x A100)')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('GB (FP16)', color='#e6edf3')
ax.set_title('MoE: Memory vs Compute cost', color='#e6edf3')
ax.legend(framealpha=0.2, fontsize=8)
ax.grid(axis='y', alpha=0.3)
ax.set_yscale('log', base=2)
plt.tight_layout()
plt.show()

## Experiments: Try These

1. **Expert count sweep:** Change `n_experts` to 4, 16, 32 and `top_k` to 1, 2, 4.
   Plot total params vs active params for each combo. Where is the sweet spot?

2. **Load balancing:** Run 1000 tokens through the router. Plot the distribution of
   expert selections. Add a simple load-balance penalty: subtract `0.1 * count[expert]`
   from the router logits before softmax. Does it help even out the distribution?

3. **Batch utilization:** For the Qwen config (128 experts, top-8), calculate exactly
   how many concurrent requests are needed to guarantee all 128 experts are active
   (worst case: each request picks completely non-overlapping experts).

## Key Takeaways

- **MoE replaces the dense FFN** with N smaller expert FFNs and a router. Only K experts are active per token.
- **Total params >> active params.** Qwen3-235B has 235B total but only 22B active per token. You need memory for all of them, but only compute for the active ones.
- **The router** is a simple linear layer that scores each expert per token. It runs per-token, per-layer.
- **Load balancing** is critical — unbalanced routing creates hot-shard problems. Training uses auxiliary losses to prevent this.
- **Sparsity shrinks with batch size.** With enough concurrent requests, all experts are active and the MoE model behaves like a dense model for compute.
- **MoE models excel at interactive, low-batch serving** where single-request latency matters most.
- Next: Day 04 — Ops:Byte Ratio & Arithmetic Intensity.

## References

- *Inference Engineering* Ch 2.2.4 (pp. 53–55) — Philip Kiely, Baseten Books 2026
- Fedus et al., "Switch Transformers" (2021)
- Jiang et al., "Mixtral of Experts" (2024)